<p align="left">
  <a href="https://colab.research.google.com/github/wgomezf/CNN_workshop/blob/main/Notebooks/gradcam.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" width="200">
  </a>
</p>

# Gradient-weighted Class Activation Mapping

Grad-CAM (https://doi.org/10.1109/ICCV.2017.74) is an explainability technique that visually highlights the regions of an image that most strongly influence a CNN's prediction. It creates a heatmap overlay that reveals what the network "looks at" when making a decision. The ResNet-50 network trained on the ImageNet dataset is used for the examples.

In [ ]:
#@title Load necessary libraries
import numpy as np                                                    # Numerical array operations
import matplotlib.pyplot as plt                                       # Data plotting/visualization
import tensorflow as tf                                               # Deep learning
import os                                                             # Interaction with the operating system
import cv2                                                            # Computer vision
import urllib.request                                                 # Open and read URLs
from tensorflow.keras.applications.resnet50 import ResNet50           # Pre-trained CNN

In [ ]:
#@title Function to read an image from a URL
def get_img_array(img_path, size=(224, 224)):
  # Fetch image from URL
  resp = urllib.request.urlopen(img_path)
  image_data = np.asarray(bytearray(resp.read()), dtype="uint8")
  img = cv2.imdecode(image_data, cv2.IMREAD_COLOR)
  # Convert BGR to RGB
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  # Resize to target size
  img = cv2.resize(img, size)
  return img

In [ ]:
#@title Function that implements the Grad-CAM technique
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):

  # Convert to float32
  img_array = tf.keras.preprocessing.image.img_to_array(img_array)
  img_array = np.expand_dims(img_array, axis=0)

  grad_model = tf.keras.models.Model(
      model.input,
      [model.get_layer(last_conv_layer_name).output, model.output])

  with tf.GradientTape() as tape:
      conv_outputs, predictions = grad_model(img_array)
      if pred_index is None:
          pred_index = tf.argmax(predictions[0])
      class_channel = predictions[:, pred_index]

  grads = tape.gradient(class_channel, conv_outputs)
  pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

  conv_outputs = conv_outputs[0]
  heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
  heatmap = tf.squeeze(heatmap)
  heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
  return heatmap.numpy(), predictions

In [ ]:
#@title Function to overlay a heatmap on the input image
def display_gradcam(img, heatmap, alpha=0.4):

  # Resize heatmap
  heatmap = cv2.resize(heatmap, (img.shape[0], img.shape[1]))
  heatmap = np.uint8(255 * (1-heatmap))
  heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

  # Superimpose
  superimposed_img = cv2.addWeighted(img, 1 - alpha, heatmap_color, alpha, 0)

  # Display
  plt.figure(figsize=(8, 4))
  plt.subplot(1, 2, 1)
  plt.imshow(img)
  plt.title('Input image')
  plt.axis('off')

  plt.subplot(1, 2, 2)
  plt.imshow(superimposed_img)
  plt.title('Grad-CAM heatmap')
  plt.axis('off')

  plt.tight_layout()
  plt.show()

In [ ]:
#@title Load the ResNet-50 network
model = ResNet50(weights='imagenet')
# model.summary()

In [ ]:
#@title Classify an image and show the Grad-CAM heatmap
# Pick an image
# img_path = 'https://siamesekittykat.com/wp-content/uploads/2025/07/Siamese-Cat.png'
img_path = 'https://c02.purpledshub.com/uploads/sites/39/2019/03/1375792762282-nt9ibl5k3vvu-5410d3b.jpg'
# img_path = 'https://muyinteresante.okdiario.com/wp-content/uploads/sites/5/2023/01/22/63cd065d4490e.jpeg'
# img_path = 'https://www.slashgear.com/img/gallery/heres-why-fords-first-production-car-was-called-the-model-t/intro-1705710312.jpg'
# img_path = 'https://cdn.britannica.com/76/116576-050-1B35C79A/Man-snowmobile-Bighorn-Mountains-Montana.jpg'
# img_path = 'https://images.squarespace-cdn.com/content/v1/657b302ad0d11e71b22b40c3/1705335532774-DAY9SFG66ATMAAZGOSVL/preview_13.normal.jpg'
# img_path = 'https://images.puppyfinder.com/Breed/1/c/b/1cb6edfcc5f4c7b9_802_image1_medium.jpg'
# img_path = 'https://www.shutterstock.com/image-photo/closeup-freshly-picked-cabbage-farmers-600nw-2671380677.jpg'
# img_path = 'https://images.stockcake.com/public/0/e/5/0e593b95-298d-454e-87e8-d1ee2f7eebf0_large/joyful-comic-reading-stockcake.jpg'
# img_path = 'https://www.shutterstock.com/image-photo/cropped-young-male-doctor-man-600nw-2717661525.jpg'
# img_path = 'https://cdn.britannica.com/80/162480-004-9DFF6461/People-beach-volleyball.jpg'
# img_path = 'https://keto-mojo.com/wp-content/uploads/2021/09/Keto-Guacamole.jpg'

# Get image tensor and Grad-CAM heatmap
IMG_SHAPE = (224, 224)
img_array = get_img_array(img_path, size=IMG_SHAPE)
heatmap, preds = make_gradcam_heatmap(img_array, model, last_conv_layer_name='conv5_block3_out')

# Predict class
class_name = tf.keras.applications.resnet.decode_predictions(preds, top=1)[0][0][1]
print('Predicted class:', class_name)

# Display results
display_gradcam(img_array, heatmap)
